# 🧪 Praktikum 10 – RAG-Evaluation & Benchmarking
**Applied Generative AI – NLP | Sommersemester 2026**

> 🎯 **Lernziele:** RAG-Systeme systematisch bewerten · Faithfulness (Faktentreue) vs. Relevanz unterscheiden · Ein automatisiertes Test-Skript entwickeln

⏱️ **Dauer:** 90 Minuten

In [ ]:
import os, sys, subprocess, shutil, time, requests, pandas as pd
MODEL = "qwen3.5:0.8b"
def install_if_missing(pkgs):
    cmd = [sys.executable, "-m", "pip", "install"] + (["--break-system-packages"] if sys.prefix == getattr(sys, "base_prefix", sys.prefix) else [])
    subprocess.check_call(cmd + pkgs)
install_if_missing(["ollama", "requests", "pandas"])
import ollama
print("✅ Setup abgeschlossen.")

## Teil 1 – Das Test-Set (Golden Dataset) ⏱️ 25 min
Wir erstellen eine Liste aus Fragen, Kontexten und Ground-Truth Antworten.

In [ ]:
test_set = [
    {
        "q": "Wie hoch ist der Mount Everest?",
        "c": "Der Mount Everest ist 8848 Meter hoch.",
        "gt": "8848 Meter"
    },
    {
        "q": "Wer schrieb Faust?",
        "c": "Goethe verfasste das Drama Faust.",
        "gt": "Johann Wolfgang von Goethe"
    }
]

def get_answer(query, context):
    prompt = f"Kontext: {context}\nFrage: {query}"
    res = ollama.chat(model=MODEL, messages=[{"role": "user", "content": prompt}])
    return res['message']['content']

results = []
for item in test_set:
    ans = get_answer(item['q'], item['c'])
    results.append({"query": item['q'], "answer": ans, "ground_truth": item['gt']})

df = pd.DataFrame(results)
print(df)

## Teil 2 – Automated LLM-as-a-Judge ⏱️ 40 min
Wir lassen ein LLM prüfen, ob die `answer` zur `ground_truth` passt.

In [ ]:
def judge_answer(query, answer, gt):
    prompt = f"""Bewerte die Korrektheit der Antwort basierend auf der Ground Truth.
    Frage: {query}
    Antwort: {answer}
    Ground Truth: {gt}
    
    Gib einen Score von 0 (falsch) bis 1 (korrekt). Antworte NUR mit der Zahl.
    """
    res = ollama.chat(model=MODEL, messages=[{"role": "user", "content": prompt}])
    try: return float(re.findall(r"\d+\.?\d*", res['message']['content'])[0])
    except: return 0.0

df['score'] = df.apply(lambda x: judge_answer(x['query'], x['answer'], x['ground_truth']), axis=1)
print(f"Durchschnittlicher Score: {df['score'].mean()}")

## Teil 3 – Faithfulness Check ⏱️ 25 min
Halluziniert das Modell? Wir prüfen, ob die Antwort NUR Informationen aus dem Kontext enthält.

In [ ]:
def check_faithfulness(context, answer):
    prompt = f"Enthält die Antwort Informationen, die NICHT im Kontext stehen?\nKontext: {context}\nAntwort: {answer}\nAntworte mit JA oder NEIN."
    res = ollama.chat(model=MODEL, messages=[{"role": "user", "content": prompt}])
    return res['message']['content']

print("Faithfulness Check:", check_faithfulness(test_set[0]['c'], "Der Mount Everest ist 8848m hoch und liegt im Himalaya."))

## 🧩 Aufgaben
1. **Diverse Dataset:** Erweitere das `test_set` auf 10 verschiedene Fragen (auch Fangfragen!).
2. **Metric Comparison:** Berechne für deine Ergebnisse zusätzlich den Jaccard-Score. Wie stark korreliert er mit dem LLM-Judge?
3. **RAGAS:** Recherchiere das Framework 'RAGAS'. Welche 3 Hauptmetriken werden dort zur Bewertung von RAG genutzt?